In [14]:
### follow this notebook fo rmultiple charge trap implementation

In [15]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
from Minimal_Model_2D import *
import time

np.set_printoptions(threshold=sys.maxsize)

In [16]:
class Minimal_Model_multiple_charge_trap_potential(twoD_Minimal_Model):

    ### overriding a H same spin method to include multiple charge trap 

    def H_same_spin(self, x_ct= np.empty(5), y_ct=np.empty(5), z_ct=np.empty(5), z_im=np.empty(5), z_qw=0, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0):

        #print('a- same spin class overridden')

        M = (hbm/self.meff_sm* self.twodOP.KxSq(kx) + hbm/self.meff_sm* self.twodOP.KySq(ky) - self.chem_pot() + 1/4/hbm*self.meff_sm*self.omega**2*self.twodOP.ysq()  +self.dis_pot_matrix(dis_mag) +self.Disorder_full(uncorr_var, lmbda, seed) )

        for i in range(x_ct.shape[0]):

            M = M + self.charge_pot_lattice(x_ct[i], y_ct[i], z_ct[i], z_im[i], z_qw)
 
        return M


In [10]:


NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 20        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

MY= 5 

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print('mu = ', MU_SM)


UNCORR_VAR = 0  ### uncorrelated variance of disorder
LMBDA = 0
SEED = 58  ## seed for the disorder

# Define the range and the size of the array
x_low = (NSC+1)*AX  # Start of the range (inclusive)
x_high = (NSC+NSM+1)*AX  # End of the range (exclusive)

y_low = 0.0  # Start of the range (inclusive)
y_high = M*AY  # End of the range (exclusive)

size = 5  # Size of the array

# Generate the array
X_CT = np.random.uniform(x_low, x_high, size)
Y_CT = np.random.uniform(y_low, y_high, size)
Z_CT = np.ones(size)*50                          ###all charg etraps assumed to be in oxide interface
Z_IM  = np.ones(size)*64 


print(X_CT)
print(Y_CT)
print(Z_CT)
print(Z_IM)


ModCT = Minimal_Model_multiple_charge_trap_potential(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

lengths= 300 100
subband energies= 0.6631720133634798 1.9895160400904393
4t= 265.2688053453919
mu =  0.7958064160361757
[4198.95620704 4272.91308592 4189.65697764 4203.57287954 4016.83405735]
[ 94.37593047   5.82061916 142.95084701 208.12081931 204.27691049]
[50. 50. 50. 50. 50.]
[64. 64. 64. 64. 64.]


In [ ]:

H = ModCT.H_sys(PHI, 'li', X_CT, Y_CT, Z_CT, Z_IM)

E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

Es = np.sort(E)/DELTA

#Ep[i,:] = Es

print(Es)

#print(H.shape)

In [6]:
ITER = 21

Ep = np.empty((ITER,K_NUM))

Z_CT = np.ones(size)*50                          ###all charge traps assumed to be in oxide interface
Z_IM  = np.ones(size)*64 

for i in range(ITER): 

        
    # Generate the array
    X_CT = np.random.uniform(x_low, x_high, size)
    Y_CT = np.random.uniform(y_low, y_high, size) 
        
    H = ModCT.H_sys(PHI, 'li', X_CT, Y_CT, Z_CT, Z_IM)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)/DELTA

    Ep[i,:] = Es

    if i%5==0:

        print(Es)

    #print(H.shape)

    
np.save("linear_spin_splitting_{}_charge_traps_{}_iter_const_z.npy".format(size,ITER), Ep)


[-0.51986372 -0.51985496  0.51985496  0.51986372]
[-0.73358369 -0.73352295  0.73352295  0.73358369]
[-0.50214377 -0.50200982  0.50200982  0.50214377]
[-0.78154483 -0.78142223  0.78142223  0.78154483]
[-0.80303611 -0.80302255  0.80302255  0.80303611]


In [17]:
Ep = np.load("linear_spin_splitting_{}_charge_traps_{}_iter_const_z.npy".format(size,ITER))



In [18]:
Ep_splitting = Ep[:,K_NUM//2+1] - Ep[:, K_NUM//2]
print(np.mean(Ep_splitting))
print(np.var(Ep_splitting))

9.544612321185509e-05
8.287178227214266e-09


In [19]:
print(Ep_splitting)

[8.75982278e-06 7.15525498e-05 3.46342460e-05 1.26808392e-04
 1.31168994e-04 6.07407226e-05 2.89087946e-04 1.37197634e-05
 2.64685787e-05 4.55278352e-05 1.33942830e-04 2.63915513e-04
 3.29693501e-04 9.80950178e-05 9.30324041e-05 1.22599304e-04
 4.90385901e-05 1.38528103e-05 3.81628313e-05 4.00016847e-05
 1.35652510e-05]
